# ห้องทดลอง: การนับจำนวนแอนติพาลินโดรม

**วิธีใช้ใน Colab**

1. อัปโหลดไฟล์ `student-lab.ipynb` นี้เข้า Colab (`File > Upload notebook`) หรือเปิด <https://colab.research.google.com> แล้วเลือก `File > Open notebook`
2. ไปที่เมนู `Runtime` (หรือ `Execution`) → `Change runtime type`
3. ช่อง *Runtime type* เลือก **Julia** แล้วกด `Save`
4. รันทีละเซลล์จากบนลงล่าง (`Shift+Enter`)

> Colab มี Julia ติดตั้งมาให้แล้ว (เวอร์ชัน 1.12.6 ในรันไทม์ 2026.10) ไม่ต้องติดตั้งอะไรเพิ่ม

เอกสารนี้ให้ **นิยาม** และ **ตัวนับแบบตรงไปตรงมา** เท่านั้น ยังไม่มีสูตรใด ๆ หน้าที่ของเราคือคิดสูตรเองแล้วเอามาทดสอบกับตัวนับนี้


## 1. นิยาม

เขียน $n$ ในฐาน $b$ เป็น $n=(n_k n_{k-1}\cdots n_0)_b$ โดย $n_k \ne 0$ และให้ $k$ เป็นดัชนีหลักซ้ายสุด

แอนติพาลินโดรม (ตาม Pongsriiam และ Saito, 2023) มีสี่ชนิด

| ชนิด | เงื่อนไข | จำนวนหลัก |
|---|---|---|
| 0 | $n_{k-i}+n_i=b-1$ สำหรับ $0\le i\le \lfloor k/2\rfloor$ | $k$ คี่ (ความยาวคู่) |
| 1 | $n_{k-i}+n_i=b-1$ สำหรับ $0\le i\le \lfloor k/2\rfloor$ | ความยาวใดก็ได้ |
| 2 | $n_{k-i}+n_i=b-1$ สำหรับ $0\le i\le \lfloor (k-1)/2\rfloor$ | ความยาวใดก็ได้ |
| 3 | $n_{k-i}\ne n_i$ สำหรับ $0\le i\le \lfloor (k-1)/2\rfloor$ | ความยาวใดก็ได้ |

**อนุสัญญาที่ใช้**: จำนวนหนึ่งหลักทุกตัว ($1\le n<b$) นับเป็นชนิด 2 และชนิด 3 เท่านั้น ไม่นับเป็นชนิด 1


In [ ]:
"""แปลง n เป็นเวกเตอร์หลักในฐาน b โดยหลักซ้ายสุดอยู่ที่ index 1"""
function digitsof(n::Integer, b::Integer)
    d = Int[]
    while n > 0
        push!(d, Int(n % b))
        n ÷= b
    end
    return reverse!(d)
end

digitsof(2026, 10)   # ตัวอย่าง: 2026 ในฐานสิบ

## 2. ตัวนับแบบตรงไปตรงมา (ใช้ได้ แต่ช้ามาก)

เขียนฟังก์ชันตรวจว่า $n$ เป็นแอนติพาลินโดรมชนิด $j$ หรือไม่ **ตามนิยามตรง ๆ**


In [ ]:
function is_anti(n::Integer, b::Integer, j::Int)
    d = digitsof(n, b)
    k = length(d) - 1          # ดัชนีหลักซ้ายสุด
    if k == 0                  # จำนวนหนึ่งหลัก
        return j == 2 || j == 3
    end
    if j == 0
        k % 2 == 0 && return false          # ชนิด 0 ต้องมีความยาวคู่
        return all(d[k + 1 - i] + d[i + 1] == b - 1 for i in 0:(k ÷ 2))
    elseif j == 1
        return all(d[k + 1 - i] + d[i + 1] == b - 1 for i in 0:(k ÷ 2))
    elseif j == 2
        return all(d[k + 1 - i] + d[i + 1] == b - 1 for i in 0:((k - 1) ÷ 2))
    elseif j == 3
        return all(d[k + 1 - i] != d[i + 1] for i in 0:((k - 1) ÷ 2))
    end
    error("j ต้องเป็น 0, 1, 2 หรือ 3")
end

# ลองดูตัวอย่างในฐานสิบ
for n in 10:30
    if is_anti(n, 10, 0) || is_anti(n, 10, 2)
        println(n, " -> ", digitsof(n, 10))
    end
end

## 3. ฟังก์ชันนับสะสม (ตัวอ้างอิงของเรา)

$A_j(m;b)$ คือจำนวนแอนติพาลินโดรมชนิด $j$ ที่มีค่า **ไม่เกิน** $m$

เวอร์ชันนี้ไล่ทีละตัวตั้งแต่ 1 ถึง $m$ จึงถูกต้องแน่นอน แต่จะช้าลงมากเมื่อ $m$ ใหญ่


In [ ]:
"""นับแอนติพาลินโดรมชนิด j ที่ <= m (วิธีตรงไปตรงมา)"""
function count_brute(m::Integer, b::Integer, j::Int)
    total = 0
    for n in 1:m
        total += is_anti(n, b, j) ? 1 : 0
    end
    return total
end

@time println("A_1(1000; 10) = ", count_brute(1000, 10, 1))
@time println("A_2(1000; 10) = ", count_brute(1000, 10, 2))

## 4. วัดความเร็ว: ดูว่า brute force ไปได้ไกลแค่ไหน

ลองเพิ่ม $m$ ขึ้นทีละสิบเท่า สังเกตว่าเวลาที่ใช้โตแบบไหน

**คำถาม**: ถ้าต้องการ $m=10^{12}$ วิธีนี้จะใช้เวลาเท่าไร?


In [ ]:
for m in (10^3, 10^4, 10^5, 10^6, 10^7)
    t = @elapsed count_brute(m, 10, 1)
    println("m = 10^", round(Int, log10(m)), "\tเวลา ", round(t, digits=4), " วินาที")
end

## 5. ถึงตาคุณ: เขียนสูตรของตัวเอง

เติมสูตรของคุณในฟังก์ชัน `count_mine` ด้านล่าง แล้วรันเซลล์ทดสอบ

ตัวทดสอบจะเทียบผลของคุณกับ `count_brute` ในหลายฐานและหลายช่วง $m$ ให้อัตโนมัติ

>**คำใบ้**: ลองเริ่มจากกรณี $m$ มีจำนวนหลักเป็น **คู่** ก่อน แล้วค่อยขยายไปกรณีคี่


In [ ]:
"""สูตรของคุณ: นับแอนติพาลินโดรมชนิด j ที่ <= m

แก้ส่วนที่เขียนว่า `return 0` ให้เป็นสูตรของคุณ"""
function count_mine(m::Integer, b::Integer, j::Int)
    m <= 0 && return 0
    # ------------------------------------------------------------------
    # เขียนสูตรของคุณตรงนี้
    # ------------------------------------------------------------------
    return 0
end

In [ ]:
"""ตัวทดสอบอัตโนมัติ: เทียบ count_mine กับ count_brute"""
function run_selftest(; bases = 2:12, mmax = 2000, types = (0, 1, 2))
    # กันความเข้าใจผิด: ถ้ายังไม่ได้เขียนสูตร อย่าเพิ่งรายงานว่าผิด
    if count_mine(1, 2, 2) == 0
        println("ยังไม่ได้เขียนสูตร")
        println("count_mine ยังคืนค่า 0 ทุกกรณี (ซึ่งผิด เพราะ 1 คือแอนติพาลินโดรมชนิด 2)")
        println("กลับไปที่เซลล์ด้านบน เติมสูตรของคุณใน count_mine แล้วรันเซลล์นี้ใหม่")
        return nothing
    end

    fails = 0
    checks = 0
    first_fail = nothing
    for b in bases
        run = Dict(j => 0 for j in types)
        for m in 1:mmax
            for j in types
                run[j] += is_anti(m, b, j) ? 1 : 0
                got = count_mine(m, b, j)
                checks += 1
                if got != run[j]
                    fails += 1
                    if first_fail === nothing
                        first_fail = (b = b, m = m, j = j, want = run[j], got = got)
                    end
                end
            end
        end
    end

    println("ตรวจ ", checks, " กรณี")
    if fails == 0
        println("ผ่านทั้งหมด")
    else
        println("ไม่ผ่าน ", fails, " กรณี")
        println("ตัวอย่างแรกที่ผิด: ", first_fail)
        println("คำแนะนำ: ตรวจว่า handle ความยาวคี่/คู่ และฐานคี่/คู่ครบหรือยัง")
    end
    return fails
end

run_selftest()

## 6. โจทย์ท้าทาย

1. **สูตรต่อความยาว**: หาจำนวนแอนติพาลินโดรมชนิด $j$ ที่มีความยาว $L$ หลักพอดี เขียนเป็น $N_j(L;b)$ แล้วทดสอบกับ brute force
2. **ขยายช่วงทดสอบ**: เมื่อสูตรถูกแล้ว ให้เปลี่ยน `run_selftest(bases = 2:36, mmax = 200000)` ดูว่ายังผ่านไหม
3. **ตรวจที่ $m$ ใหญ่**: เขียนตัวตรวจที่ใช้ *ผลต่าง* $A_j(m)-A_j(m-1)$ ซึ่งต้องเท่ากับ 1 เมื่อ $m$ เป็นแอนติพาลินโดรม และ 0 เมื่อไม่ใช่ วิธีนี้ทำให้ตรวจที่ $m=10^{18}$ ได้โดยไม่ต้องไล่ทีละค่า
4. **ชนิด 3**: ชนิดนี้ไม่อยู่ในเงื่อนไข "ครึ่งแรกกำหนดครึ่งหลัง" ลองคิดว่าจะนับอย่างไร

## 7. เช็กลิสต์ก่อนส่งงาน

- [ ] สูตรผ่าน `run_selftest` ครบทุกฐาน 2 ถึง 36
- [ ] อธิบายได้ว่าทำไมสูตรถึงถูก ไม่ใช่แค่ "รันแล้วผ่าน"
- [ ] เขียน proof sketch ของกรณีความยาวคี่ได้
- [ ] ระบุจุดที่สูตรใช้ไม่ได้ (ถ้ามี)
